# Expliquer un détecteur d'intrusion (UNSW-NB15)

Projet du module Machine Learning pour la cyber-sécurité, cahier des charges n° 1.

Binôme : *à compléter*

Ce notebook est à la fois le code et le rapport. Il s'exécute de haut en bas, dans l'ordre, sans intervention manuelle.

## 0. Mise en place

In [ ]:
import sys
from pathlib import Path

# Le notebook est dans notebooks/, le code partagé dans src/ à la racine du projet.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import time

import numpy as np
import pandas as pd
import sklearn, xgboost, shap

from src import config
import matplotlib.pyplot as plt

from src import data as D
from src.preprocessing import build_preprocessor, numeric_columns
from src import models as M

config.set_seed()
print("Python", sys.version.split()[0])
print("numpy", np.__version__, "| pandas", pd.__version__, "| scikit-learn", sklearn.__version__,
      "| xgboost", xgboost.__version__, "| shap", shap.__version__)
print("Graine :", config.SEED)

## 1. Problématique

## 2. Données

### 2.1 Description du jeu

Nous utilisons UNSW-NB15 (Moustafa et Slay, 2015), produit par le Cyber Range Lab d'UNSW Canberra. Le trafic a été généré sur un banc d'essai mêlant trafic normal réaliste et attaques synthétiques, puis découpé en flux réseau. Chaque flux est décrit par 42 variables : durée, octets et paquets dans chaque sens, TTL, fenêtres TCP, délais, ainsi que des compteurs de connexions récentes (variables `ct_*`). Trois variables sont textuelles : le protocole (`proto`), le service applicatif (`service`) et l'état de la connexion (`state`).

Les auteurs fournissent deux fichiers déjà séparés, un fichier d'entraînement et un fichier de test. Chaque ligne porte une étiquette binaire (`label`, 1 = attaque) et la famille d'attaque (`attack_cat`). Notre modèle principal est binaire. La famille ne sert qu'à l'analyse, jamais comme variable d'entrée.

In [ ]:
raw_train, raw_test = D.load_raw()

for nom, df in [("entraînement", raw_train), ("test", raw_test)]:
    print(f"Fichier {nom} : {len(df):,} lignes, {df.shape[1]} colonnes, "
          f"{df[config.TARGET].mean():.1%} d'attaques")

print("\nTypes de colonnes :", raw_train.dtypes.value_counts().to_dict())

In [ ]:
familles = pd.DataFrame({
    "entraînement": raw_train[config.ATTACK_CAT].value_counts(),
    "test": raw_test[config.ATTACK_CAT].value_counts(),
})
familles

Deux remarques dès cette étape.

Les attaques sont **majoritaires** dans les deux fichiers, alors que dans un vrai réseau elles sont rares. Les auteurs n'ont conservé qu'un échantillon du trafic normal. Conséquence pour l'évaluation : la PR-AUC d'un classifieur aléatoire est égale à la proportion d'attaques, il faudra donc toujours lire nos PR-AUC par rapport à ce niveau de référence.

Les familles sont très déséquilibrées entre elles (Generic : 40 000 flux d'entraînement, Worms : 130). Les familles rares seront à surveiller dans l'analyse des erreurs.

### 2.2 Nettoyage

Les fichiers ne contiennent ni valeur manquante, ni valeur infinie, ni valeur négative :

In [ ]:
for nom, df in [("entraînement", raw_train), ("test", raw_test)]:
    num = df.select_dtypes("number")
    print(f"{nom} : manquantes = {int(df.isna().sum().sum())}, "
          f"infinies = {int(np.isinf(num).sum().sum())}, "
          f"négatives = {int((num < 0).sum().sum())}")

En revanche, quatre problèmes demandent une décision.

**La colonne `id` révèle l'étiquette.** Les lignes sont rangées par `id`, et cet ordre suit l'étiquette : la proportion d'attaques par quart du fichier d'entraînement passe de 0 % à 100 %. Un modèle qui verrait `id` apprendrait l'ordre du fichier et non le trafic. Nous la supprimons.

In [ ]:
quarts = pd.qcut(raw_train["id"], 4, labels=["1er quart", "2e quart", "3e quart", "4e quart"])
raw_train.groupby(quarts, observed=True)[config.TARGET].mean().round(3).rename("part d'attaques")

**Doublons.** Beaucoup de lignes sont des copies exactes d'une autre (mêmes variables, même étiquette). Les garder fausserait la validation croisée, car des copies d'un même flux se retrouveraient des deux côtés d'un pli. Nous gardons une seule ligne par copie. Nous supprimons les doublons **avant** tout découpage, sinon ils pourraient fuir entre entraînement et validation.

**Contradictions.** Certains vecteurs de variables apparaissent à la fois avec l'étiquette « normal » et l'étiquette « attaque ». Aucun modèle ne peut classer correctement les deux. Ces lignes illustrent le chevauchement de classes décrit par Zoghi et Serpen (2021) dans leur analyse du jeu. Nous les retirons toutes.

La même fonction est appliquée aux deux fichiers.

In [ ]:
train_clean, rapport_train = D.clean(raw_train)
test_clean, rapport_test = D.clean(raw_test)
pd.DataFrame({"entraînement": rapport_train, "test": rapport_test})

**Recouvrement entre les deux fichiers.** Des flux du fichier de test ont exactement les mêmes variables qu'un flux du fichier d'entraînement. Un modèle les classerait juste par simple mémoire, ce qui mesurerait la mémoire et non la généralisation. Nous les retirons du test. La comparaison se fait sur les variables seulement, quelle que soit l'étiquette.

In [ ]:
test_final, rapport_recouvrement = D.remove_overlap(test_clean, raw_train)
rapport_recouvrement

**Valeurs textuelles inconnues.** La variable `state` prend dans le test deux valeurs absentes de l'entraînement (`ACC` et `CLO`). L'encodage one-hot devra les accepter sans erreur : une valeur inconnue sera codée par des zéros. Par ailleurs, `service` vaut `-` pour environ la moitié des flux. Ce n'est pas une valeur manquante mais une vraie catégorie, « aucun service reconnu », que nous conservons telle quelle.

In [ ]:
for col in config.CATEGORICAL:
    inconnues = sorted(set(test_final[col]) - set(train_clean[col]))
    print(f"{col} : {train_clean[col].nunique()} valeurs en entraînement, inconnues dans le test : {inconnues}")

### 2.3 Découpage entraînement / validation / test

**Pas de découpage chronologique possible.** Le cahier des charges demande de respecter l'ordre du temps quand les données sont horodatées. Les fichiers de partition d'UNSW-NB15 ne contiennent pas les colonnes de temps (`stime`, `ltime`) du jeu complet. Nous conservons donc la séparation officielle des auteurs : le fichier de test nettoyé sert de jeu de test.

**Validation.** Le fichier d'entraînement nettoyé est coupé en 80 % pour l'entraînement et 20 % pour la validation. Le découpage est stratifié sur la famille d'attaque, pour que les familles rares soient présentes des deux côtés dans les mêmes proportions. Comme « Normal » est une des familles, la proportion d'attaques est conservée aussi.

Rôle de chaque jeu :
- l'**entraînement** sert à ajuster les modèles et, par validation croisée, à régler leurs hyperparamètres ;
- la **validation** sert à comparer les modèles réglés et à choisir un seuil de décision ;
- le **test** n'est utilisé qu'une seule fois, pour les résultats finaux.

In [ ]:
train, val = D.split_train_val(train_clean)
test = test_final

resume = pd.DataFrame({
    nom: {"flux": len(df), "part d'attaques": round(df[config.TARGET].mean(), 3),
          "Worms": int((df[config.ATTACK_CAT] == "Worms").sum())}
    for nom, df in [("entraînement", train), ("validation", val), ("test", test)]
}).T
resume = resume.astype({"flux": int, "Worms": int})
resume

In [ ]:
# Vérifications : aucun vecteur de variables partagé entre les trois jeux.
cols = D.feature_columns(train)
def communs(a, b):
    return len(a[cols].drop_duplicates().merge(b[cols].drop_duplicates()))
print("entraînement et validation :", communs(train, val))
print("entraînement et test       :", communs(train, test))
print("validation et test         :", communs(val, test))

In [ ]:
part = pd.DataFrame({nom: df[config.ATTACK_CAT].value_counts(normalize=True)
                     for nom, df in [("entraînement", train), ("validation", val), ("test", test)]})
ax = part.sort_values("entraînement").plot.barh(figsize=(8, 5))
ax.set_xlabel("part des flux")
ax.set_title("Répartition des familles après nettoyage")
plt.tight_layout()
plt.savefig(config.FIGURES / "familles_apres_nettoyage.png", dpi=150)
plt.show()

Après nettoyage, la part d'attaques baisse nettement dans les deux fichiers : beaucoup de doublons étaient des attaques, en particulier de la famille Generic. Le test garde une répartition différente de l'entraînement : beaucoup plus de trafic normal (64 % contre 51 %), moins d'Exploits et de Fuzzers, un peu plus de Generic. C'est un écart réel entre les deux fichiers officiels, dont il faudra tenir compte en lisant les résultats sur le test.

**Prétraitement.** L'encodage one-hot des variables textuelles et, pour la régression logistique seulement, la standardisation des variables numériques sont placés dans un `Pipeline` scikit-learn avec chaque modèle (voir `src/preprocessing.py`). Ils sont donc ajustés sur l'entraînement uniquement, et à l'intérieur de chaque pli pendant la validation croisée. Aucune opération apprise ne voit la validation ni le test.

In [ ]:
X_train, y_train, fam_train = D.get_xy(train)
X_val, y_val, fam_val = D.get_xy(val)
X_test, y_test, fam_test = D.get_xy(test)

# Contrôle : le prétraitement s'ajuste sur l'entraînement et s'applique au test sans erreur.
prep = build_preprocessor(numeric_columns(X_train), scale=True).fit(X_train)
print("Variables après encodage :", prep.transform(X_train).shape[1])
print("Forme du test encodé     :", prep.transform(X_test).shape)

## 3. Modèles

### Protocole commun

Les trois modèles suivent le même protocole, pour que la comparaison soit équitable.

- Chaque modèle est un `Pipeline` : prétraitement (section 2.3) puis classifieur. Le prétraitement est donc réajusté dans chaque pli de validation croisée.
- Les hyperparamètres sont réglés par **validation croisée stratifiée à 5 plis sur le jeu d'entraînement seulement**. La validation et le test ne servent pas au réglage.
- Le critère de réglage est la **PR-AUC** (`average_precision`). Elle juge le classement des flux par score sur tous les seuils, sans dépendre du seuil de 0,5, et elle reste adaptée au déséquilibre des classes.
- La graine est fixée à 42 partout (découpage des plis, tirage des combinaisons, aléa interne des modèles).

Pour la régression logistique, les cinq valeurs possibles sont toutes essayées (recherche en grille). Pour la forêt et XGBoost, les combinaisons possibles se comptent par milliers. Nous en tirons 12 au hasard (recherche aléatoire), soit 60 entraînements par modèle avec les 5 plis. C'est un compromis assumé avec le temps de calcul sur processeur.

In [ ]:
pipelines = M.make_pipelines(numeric_columns(X_train))
recherches = {}
duree_recherche = {}

def regler(nom):
    debut = time.perf_counter()
    recherches[nom] = M.tune(nom, pipelines[nom], X_train, y_train)
    duree_recherche[nom] = time.perf_counter() - debut
    moyenne, ecart = M.cv_summary(recherches[nom])
    print(f"{nom} : PR-AUC en validation croisée = {moyenne:.4f} ± {ecart:.4f} "
          f"(recherche : {duree_recherche[nom] / 60:.1f} min)")
    print("Meilleurs réglages :", M.short_params(recherches[nom].best_params_))

### 3.1 Baseline : régression logistique

La régression logistique attribue un poids à chaque variable, additionne les contributions et transforme la somme en probabilité. Elle ne peut tracer qu'une frontière linéaire entre les classes. C'est ce qui en fait une bonne référence : les modèles plus complexes doivent la battre nettement pour justifier leur coût. Les variables numériques sont standardisées pour elle. Le seul hyperparamètre réglé est `C`, l'inverse de la force de régularisation.

In [ ]:
regler("Régression logistique")

### 3.2 Forêt aléatoire

Une forêt aléatoire combine des centaines d'arbres de décision. Chaque arbre est entraîné sur un échantillon tiré au hasard des lignes et ne considère qu'une partie des variables à chaque division. Le vote moyen des arbres corrige une grande partie des erreurs individuelles. Réglages explorés : nombre d'arbres, profondeur maximale, taille minimale des feuilles et part des variables considérées à chaque division.

In [ ]:
regler("Forêt aléatoire")

### 3.3 XGBoost

XGBoost construit aussi des arbres, mais l'un après l'autre : chaque nouvel arbre corrige les erreurs des précédents (gradient boosting). Réglages explorés : nombre d'arbres, profondeur, taux d'apprentissage, part des lignes et des variables tirées pour chaque arbre, et poids minimal d'une feuille.

In [ ]:
regler("XGBoost")

### 3.4 Comparaison

Le tableau rassemble, pour chaque modèle réglé :

- la PR-AUC moyenne en validation croisée et son écart-type entre plis ;
- la PR-AUC et le F1 (au seuil par défaut de 0,5) sur le **jeu de validation**, que les modèles n'ont jamais vu ;
- le temps d'entraînement sur tout le jeu d'entraînement, sur un seul cœur pour que les trois modèles soient mesurés dans les mêmes conditions ;
- le temps d'inférence pour 1 000 flux.

Le niveau de référence de la PR-AUC est la proportion d'attaques : environ 0,49 en validation. Le test n'est toujours pas utilisé ; l'évaluation complète fait l'objet de la section 4.

In [ ]:
from sklearn.metrics import average_precision_score, f1_score

lignes = {}
for nom, recherche in recherches.items():
    modele = recherche.best_estimator_
    score_val = modele.predict_proba(X_val)[:, 1]
    cv_moy, cv_ecart = M.cv_summary(recherche)
    t_fit, t_pred = M.time_model(modele, X_train, y_train, X_val)
    lignes[nom] = {
        "PR-AUC CV (moyenne)": round(cv_moy, 4),
        "PR-AUC CV (écart-type)": round(cv_ecart, 4),
        "PR-AUC validation": round(average_precision_score(y_val, score_val), 4),
        "F1 validation (seuil 0,5)": round(f1_score(y_val, score_val >= 0.5), 4),
        "entraînement (s)": round(t_fit, 1),
        "inférence (ms / 1 000 flux)": round(t_pred, 2),
    }

comparaison = pd.DataFrame(lignes).T
print(f"Niveau de référence de la PR-AUC en validation : {y_val.mean():.3f}")
comparaison

In [ ]:
reglages = {nom: {k: str(v) for k, v in M.short_params(r.best_params_).items()}
            for nom, r in recherches.items()}
pd.DataFrame(reglages).T.fillna("")

**Lecture des résultats.**

Les trois modèles sont très loin au-dessus du niveau de référence (PR-AUC de 0,488 pour un classement au hasard en validation). Les scores de validation croisée et ceux du jeu de validation sont presque identiques, avec des écarts-types entre plis très faibles. Les modèles ne sur-apprennent donc pas leurs plis d'entraînement, et le réglage est stable.

**Les arbres battent nettement la baseline.** La régression logistique atteint une PR-AUC de 0,959 en validation, la forêt et XGBoost 0,985. Exprimé en erreur restante (1 − PR-AUC), les modèles d'arbres la réduisent d'environ deux tiers, de 0,041 à 0,015. Le F1 au seuil de 0,5 passe de 0,904 à 0,933. Les relations entre variables et attaque ne sont donc pas linéaires, ce que les arbres savent capter. La meilleure valeur de `C` pour la régression logistique est à la limite de la grille (100). Un essai séparé avec `C` de 10 à 10 000 donne la même PR-AUC à 0,0002 près : la régularisation ne joue presque pas, c'est la forme linéaire du modèle qui le limite.

**La forêt et XGBoost ne se départagent pas en performance.** L'écart de PR-AUC en validation (0,0002) est plus petit que l'écart-type entre plis (environ 0,001). Conclure que l'un est meilleur serait lire du bruit. La meilleure profondeur de XGBoost (10) est aussi la plus grande valeur explorée ; l'étendre pourrait gagner quelques millièmes, sans changer la conclusion vu la proximité des deux modèles.

**Ils se départagent en coût.** Sur un seul cœur, la forêt demande environ 200 secondes d'entraînement contre une quinzaine pour XGBoost, et son inférence est environ trois fois plus lente (35 ms contre 10 ms pour 1 000 flux). Les trois modèles restent largement compatibles avec une détection quasi temps réel : moins de 0,04 ms par flux. Les temps dépendent de la machine ; ce sont les rapports entre modèles qui comptent.

**Prudence.** Des scores aussi élevés sont précisément ce que ce sujet nous demande de questionner. La section 2 a déjà montré qu'une seule variable, `sttl`, sépare presque parfaitement certaines valeurs de trafic normal et d'attaque. Avant de croire que ces modèles reconnaissent des comportements d'attaque, il faudra vérifier sur quoi ils s'appuient (section 5).

In [ ]:
# Les modèles réglés, réutilisés dans les sections suivantes.
modeles = {nom: r.best_estimator_ for nom, r in recherches.items()}

## 4. Évaluation

### 4.1 Matrices de confusion

### 4.2 Précision, rappel, F1 par classe

### 4.3 Courbes précision-rappel et PR-AUC

### 4.4 Coût des faux positifs et des faux négatifs

## 5. Interprétabilité

### 5.1 Trois mesures d'importance (forêt, permutation, SHAP)

### 5.2 Cinq alertes expliquées (2 VP, 2 FP, 1 FN)

### 5.3 Audit de plausibilité des dix variables principales

### 5.4 Ablation des variables suspectes

### 5.5 Stabilité des explications (graine, découpage)

## 6. Analyse critique

Le modèle apprend-il des comportements d'attaque ou des particularités du jeu ?

## 7. Limites

## 8. Références